In [ ]:
pip install pymongo twython pandas

In [7]:
"""
Ejemplo de importación de Social Media en Mongo DB

Importación de twits en una base de datos Mongo.


ANTES DE EJECUTAR ESTE SCRIPT, ES NECESARIO:
    1. Tener instancia en la nube de MongoDB Atlas, MongoDB instalado en local o en un servidor propio.

EL CÓDIGO SE DIVIDO EN 3 PARTES
    1. Importación de los paquetes de python necesarios
    2. Configurar la base de datos Mongo y las colecciones
    3. Ejemplos de manipulación
"""


###### PARTE 1: Importación de los paquetes de python necesarios  ######
import pandas as pd
import pymongo #Necesario instalarlo la primera vez de forma aislada: pip install Twython
from twython import Twython #Necesario instalarlo la primera vez de forma aislada: pip install Twython
from datetime import datetime as dt
from getpass import getpass

###### PARTE 2: Configurar la base de datos Mongo y las colecciones ######
#Establecimiento Conexión a MongoDB Atlas

# Datos de ejemplo, necesario modificar por vuestra instancia en MongoDB Atlas o local, la conexión no se muestra, se rellena en el cuadro de dialogo
dbStringConnection = getpass("Pega la URI de conexión a MongoDB Atlas (no se mostrará): ").strip()

dbName = 'Twitter'
dbCollectionA = 'tweetsAccount'
dbCollectionT = 'samples' #'tweets'

client = pymongo.MongoClient(dbStringConnection)

# Definición de la base de datos MongoDB
db = client[dbName]

# Colección accounts en la base de datos para las cuentas de twitter
accounts = db[dbCollectionA]

# Colección donde tenemos los tweets
tweets = db[dbCollectionT]




Pega la URI de conexión a MongoDB Atlas (no se mostrará): ··········


In [8]:
'''
    4.1.- En la colección de cuentas de twitter, tener los campos amigos y tweets enviados,
        cargar los datos correspondientes mediante consulta mongodb + código python.
'''

pipeline_calcula_tweets_amigos = [
    {
        '$group': {
            '_id': '$user.screen_name',
            'c_total_tweets': {'$sum': '$user.statuses_count'},
            'c_friends': {'$sum': '$user.friends_count'}
        }
    },
    {
        '$addFields': {
            'new_total_tweets': '$c_total_tweets',
            'new_friends': '$c_friends'
        }
    }
]

for org in tweets.aggregate(pipeline_calcula_tweets_amigos):
    print(org)
    accounts.update_one(
        {
            'Twitter_handle': org['_id']
        },
        {
            '$set': { "num_tweets_enviados": org['new_total_tweets']},
            '$set': { "num_amigos": {'$push': {'$each': org['new_friends']}}}
        }
    )

{'_id': 'elmundoes', 'c_total_tweets': 407295000, 'c_friends': 1403000, 'new_total_tweets': 407295000, 'new_friends': 1403000}
{'_id': 'unicomplutense', 'c_total_tweets': 15247121, 'c_friends': 5089685, 'new_total_tweets': 15247121, 'new_friends': 5089685}
{'_id': 'RafaelNadal', 'c_total_tweets': 3194000, 'c_friends': 149000, 'new_total_tweets': 3194000, 'new_friends': 149000}
{'_id': 'JuanLuisGuerra', 'c_total_tweets': 1649694, 'c_friends': 65868, 'new_total_tweets': 1649694, 'new_friends': 65868}
{'_id': 'valenciacf', 'c_total_tweets': 103915668, 'c_friends': 168324, 'new_total_tweets': 103915668, 'new_friends': 168324}
{'_id': 'el_pais', 'c_total_tweets': 741943000, 'c_friends': 849000, 'new_total_tweets': 741943000, 'new_friends': 849000}
{'_id': 'LaVanguardia', 'c_total_tweets': 650271000, 'c_friends': 706000, 'new_total_tweets': 650271000, 'new_friends': 706000}
{'_id': 'LuisFonsi', 'c_total_tweets': 12317000, 'c_friends': 320000, 'new_total_tweets': 12317000, 'new_friends': 3200

In [10]:
'''
4.2.- En la colección de tweets, calcular la antigüedad para cada tweet en función de la fecha actual considerando antigüedad 0 el día de hoy
y sumando +1 por cada día transcurrido. Incluir en el mismo documento del tweet un nuevo campo que se llamará antiguedad_dias con esa antigüedad calculada.
'''

# obtenemos la fecha de hoy para calcular la antiguedad
hoy = dt.now()

# recorremos la tabla completa
for tweet in tweets.find():

    # Recuperamos la fecha de hoy y la del tweet para calcular cuantos dias han pasado
    fecha_pub = dt.strptime(tweet['created_at'], "%a %b %d %H:%M:%S +0000 %Y")
    antiguedad = (hoy - fecha_pub).days

    #actualizamos el tweet con la nueva columna
    result = tweets.update_many(
        {'id': tweet['id']},
        {
            '$set': { "antiguedad": antiguedad},
            '$set': { "date_pub": fecha_pub},
            '$set': { "day_of_week_pub": fecha_pub.strftime('%A')} # Para el ejercicio 5.7 añadimos la columna del día de la semana de publicacion
        }
    )

    print(result)


Se truncaron las últimas líneas 5000 del resultado de transmisión.
UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000027f'), 'opTime': {'ts': Timestamp(1795442362, 2), 't': 639}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1795442362, 2), 'signature': {'hash': b'\xd0IA`\x1e\xaf\x03?\x7f\x02\t\xbd\xd0\x8ah P_\xe0{', 'keyId': 7677990482462375951}}, 'operationTime': Timestamp(1795442362, 2), 'updatedExisting': True}, acknowledged=True)
UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000027f'), 'opTime': {'ts': Timestamp(1795442362, 3), 't': 639}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1795442362, 3), 'signature': {'hash': b'\xd0IA`\x1e\xaf\x03?\x7f\x02\t\xbd\xd0\x8ah P_\xe0{', 'keyId': 7677990482462375951}}, 'operationTime': Timestamp(1795442362, 3), 'updatedExisting': True}, acknowledged=True)
UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000027f'), 'opTime': {'ts': Timestamp(1795442362

CursorNotFound: cursor id 4915589872606952863 not found, full error: {'ok': 0.0, 'errmsg': 'cursor id 4915589872606952863 not found', 'code': 43, 'codeName': 'CursorNotFound', '$clusterTime': {'clusterTime': Timestamp(1795442995, 3), 'signature': {'hash': b'\xa9}o\xdeS:\x8f\xec%\x7f\xad\xe2\x14\x99\x8dqz\xdea\x91', 'keyId': 7677990482462375951}}, 'operationTime': Timestamp(1795442995, 3)}

In [9]:
'''
4.3.- En la colección de tweets, calcular la antigüedad de cada tweet relativa
con la fecha de creación de la cuenta más antigua
 de la colección actual de datos. Considerando antigüedad 0 si fue
 enviado el mismo día de creación y sumando +1 por cada día transcurrido
 desde entonces en función de la fecha del tweet.
Incluir en el mismo documento del tweet un nuevo campo se llamará
frescura_relativa_dias.
'''
min_created_at_user = ""
for t in tweets.find():
    fecha = dt.strptime(t['user']['created_at'], "%a %b %d %H:%M:%S +0000 %Y")
    if (min_created_at_user == '' or fecha < min_created_at_user):
        min_created_at_user = fecha
        print(min_created_at_user)

for tweet in tweets.find():
    # Calculamos la frescura_relativa_dias respecto de la fecha de la cuenta más antigua
    fecha_pub = dt.strptime(tweet['created_at'], "%a %b %d %H:%M:%S +0000 %Y")
    frescura_relativa_dias = (fecha_pub - min_created_at_user).days

    result = tweets.update_one(
        {'id': tweet['id']},
        {'$set': { "frescura_relativa_dias": frescura_relativa_dias}}
    )

    print(result)

Se truncaron las últimas líneas 5000 del resultado de transmisión.
UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000027f'), 'opTime': {'ts': Timestamp(1795441790, 2), 't': 639}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1795441790, 2), 'signature': {'hash': b'\xb1O\xf1\xce(\x89\xda\xfe\xcb\x85\xa1\xde\x7f=#\x9d\x96;\xc0\x93', 'keyId': 7677990482462375951}}, 'operationTime': Timestamp(1795441790, 2), 'updatedExisting': True}, acknowledged=True)
UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000027f'), 'opTime': {'ts': Timestamp(1795441790, 2), 't': 639}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1795441790, 2), 'signature': {'hash': b'\xb1O\xf1\xce(\x89\xda\xfe\xcb\x85\xa1\xde\x7f=#\x9d\x96;\xc0\x93', 'keyId': 7677990482462375951}}, 'operationTime': Timestamp(1795441790, 2), 'updatedExisting': True}, acknowledged=True)
UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000027f'), 'opTime':

In [11]:
###### PARTE 3: Exemplos de manipulación. ######

pipeline_promedio_hashtags = [
    {
        '$group': {
        '_id': '$user.screen_name',
        'total_hashtags': {"$sum": {"$size": "$entities.hashtags"}},
        'total_tweets': {'$sum': 1},
        }
    },
    {
        '$addFields': {
            'promedio_hashtags': {
                '$divide': ['$total_hashtags', '$total_tweets']
            }
        }
    }
]

for org in tweets.aggregate(pipeline_promedio_hashtags):
    print(org)
    accounts.update_one(
        {
            'Twitter_handle': org['_id']
        },
        {
            "$set":{"promedio_hashtags": org['promedio_hashtags']}
        }
    )

{'_id': 'LaVanguardia', 'total_hashtags': 94, 'total_tweets': 1000, 'promedio_hashtags': 0.094}
{'_id': 'el_pais', 'total_hashtags': 15, 'total_tweets': 1000, 'promedio_hashtags': 0.015}
{'_id': 'La_UPM', 'total_hashtags': 950, 'total_tweets': 1000, 'promedio_hashtags': 0.95}
{'_id': 'elmundoes', 'total_hashtags': 102, 'total_tweets': 1000, 'promedio_hashtags': 0.102}
{'_id': 'sanchezcastejon', 'total_hashtags': 177, 'total_tweets': 1000, 'promedio_hashtags': 0.177}
{'_id': 'unicomplutense', 'total_hashtags': 632, 'total_tweets': 997, 'promedio_hashtags': 0.633901705115346}
{'_id': 'valenciacf', 'total_hashtags': 1193, 'total_tweets': 996, 'promedio_hashtags': 1.1977911646586346}
{'_id': 'LuisFonsi', 'total_hashtags': 345, 'total_tweets': 1000, 'promedio_hashtags': 0.345}
{'_id': 'RafaelNadal', 'total_hashtags': 610, 'total_tweets': 1000, 'promedio_hashtags': 0.61}
{'_id': 'uc3m', 'total_hashtags': 896, 'total_tweets': 1000, 'promedio_hashtags': 0.896}
{'_id': 'JuanLuisGuerra', 'total_

In [16]:
# Obtener la hora actual en formato UTC
import datetime

current_time = datetime.datetime.now()

# Obtener documentos de la colección de Tweets
documents = tweets.find()

# Iterar a través de los documentos
for document in documents:
	# Extraer la fecha de creación del tweet y convertirla a un objeto datetime
	created_at_str = document['created_at']
	created_at = datetime.datetime.strptime(created_at_str, "%a %b %d %H:%M:%S %z %Y").replace(tzinfo=None)

  # Calcular diferencia fechas del tweet en minutos
	diferencia = (current_time - created_at).total_seconds()/60

  # Actualizar el documento con el campo de diferencia
	document['DiferenciaMinutos'] = int(diferencia)
	tweets.update_one({'_id': document['_id']}, {'$set': {'DiferenciaMinutos': document['DiferenciaMinutos']}})

	# Imprimir ID del tweet y su diferencia
	print(f"ID del tweet: {document['_id']}, DiferenciaMinutos: {document['DiferenciaMinutos']} minutos")

Se truncaron las últimas líneas 5000 del resultado de transmisión.
ID del tweet: 644cebc4fc978618d25533c2, DiferenciaMinutos: 1885233 minutos
ID del tweet: 644cebc6fc978618d25533cb, DiferenciaMinutos: 1885260 minutos
ID del tweet: 644cebc6fc978618d25533ce, DiferenciaMinutos: 1885274 minutos
ID del tweet: 644cebc9fc978618d25533e4, DiferenciaMinutos: 1885384 minutos
ID del tweet: 644cebcbfc978618d25533fb, DiferenciaMinutos: 1885463 minutos
ID del tweet: 644cebcdfc978618d2553404, DiferenciaMinutos: 1885555 minutos
ID del tweet: 644cebcdfc978618d2553409, DiferenciaMinutos: 1885605 minutos
ID del tweet: 644cebcdfc978618d255340a, DiferenciaMinutos: 1885635 minutos
ID del tweet: 644cebcdfc978618d255340b, DiferenciaMinutos: 1885645 minutos
ID del tweet: 644cebd3fc978618d2553433, DiferenciaMinutos: 1878871 minutos
ID del tweet: 644cebd7fc978618d2553455, DiferenciaMinutos: 1879303 minutos
ID del tweet: 644cebd9fc978618d255346a, DiferenciaMinutos: 1879555 minutos
ID del tweet: 644cebd0fc978618d25